In [ ]:
import os.path
if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Instrumentation/SANGRITA/Alan/GRB 250221A"

In [ ]:
!pip install photutils

In [ ]:
from astropy.io import fits

hdul = fits.open("20250221T033556C1o.fits.fz")

In [ ]:
print(hdul.info())

In [ ]:
print(repr(hdul[1].header))

Print the RA, Dec, and uncertainty of the GRB alert.

In [ ]:
print(hdul[1].header["ALRA"], hdul[1].header["ALDE"], hdul[1].header["ALUN"])

In [ ]:
print(hdul[1].data[2024,1100])

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(hdul[1].data, vmin=500, vmax=700)
plt.colorbar()
plt.show()

In [ ]:
import math
import warnings

import numpy as np

import astropy.stats
import astropy.visualization

import scipy.ndimage

import matplotlib.pyplot as plt

def sigma_clipped_stats(data, sigma=3.0, axis=None):
    """
    Return sigma-clipped statistics of the given data.

    This behaves exactly as:

        astropy.stats.sigma_clipped_stats(
            data, sigma=sigma, axis=axis, cenfunc="median", stdfunc="mad_std"
        )

    except that it converts all ndarrays to float32 before returning them.

    Furthermore, for the common case of clipping a stack of 2D arrays, it does
    so row by row, which is much more efficient in terms of memory use.

    :param data: The data of which to calculate the statistics.
    :param sigma: The number of standard deviations for the upper and lower
        clipping limits. Defaults to 3.0
    :param axis: The axis along with to clip the data. Defaults to None.
    :return: The mean, median, and standard deviation of the data.
    """

    if not isinstance(data, np.ndarray):
        data = np.array(data)

    with warnings.catch_warnings():

        warnings.simplefilter("ignore", Warning)

        if axis == 0 and len(data.shape) == 3:

            ny = data.shape[1]
            nx = data.shape[2]

            meanimage = np.full([ny, nx], np.nan, dtype="float32")
            medianimage = np.full([ny, nx], np.nan, dtype="float32")
            sigmaimage = np.full([ny, nx], np.nan, dtype="float32")

            for iy in range(ny):
                meanrow, medianrow, sigmarow = astropy.stats.sigma_clipped_stats(
                        data[:, iy, :],
                        sigma=sigma,
                        axis=0,
                        cenfunc="median",
                        stdfunc="mad_std",
                    )

                meanimage[iy, :] = meanrow
                medianimage[iy, :] = medianrow
                sigmaimage[iy, :] = sigmarow

            mean = meanimage
            median = medianimage
            sigma = sigmaimage

        else:

            mean, median, sigma = astropy.stats.sigma_clipped_stats(
                data, sigma=sigma, axis=axis, cenfunc="median", stdfunc="mad_std"
            )

    if isinstance(mean, np.ndarray):
        mean = mean.astype("float32")
    if isinstance(median, np.ndarray):
        median = median.astype("float32")
    if isinstance(sigma, np.ndarray):
        sigma = sigma.astype("float32")

    return mean, median, sigma


def clippedmean(data, sigma=3.0, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", Warning)
        mean, median, sigma = sigma_clipped_stats(data, sigma=sigma, axis=axis)
    return mean

def dooverscan(name, header, data):
    # Currently only works for the 4kx4k window with binning 1.
    binning = int(header["SDTBN"])
    if data.shape[0] == (4096 // binning) and data.shape[1] == (4192 // binning):
        p0 = clippedmean(
            data[(0 // binning) : (2048 // binning), (0 // binning) : (48 // binning)],
            sigma=3,
        )
        p1 = clippedmean(
            data[
                (0 // binning) : (2048 // binning),
                (4144 // binning) : (4192 // binning),
            ],
            sigma=3,
        )
        p2 = clippedmean(
            data[
                (2048 // binning) : (4096 // binning), (0 // binning) : (48 // binning)
            ],
            sigma=3,
        )
        p3 = clippedmean(
            data[
                (2048 // binning) : (4096 // binning),
                (4144 // binning) : (4192 // binning),
            ],
            sigma=3,
        )
        print(
            "%s: removing overscan levels of %.2f, %.2f, %.2f, and %.2f DN."
            % (name, p0, p1, p2, p3)
        )
        data[
            (0 // binning) : (2048 // binning), (0 // binning) : (2096 // binning)
        ] -= p0
        data[
            (0 // binning) : (2048 // binning), (2096 // binning) : (4192 // binning)
        ] -= p1
        data[
            (2048 // binning) : (4096 // binning), (0 // binning) : (2096 // binning)
        ] -= p2
        data[
            (2048 // binning) : (4096 // binning), (2096 // binning) : (4192 // binning)
        ] -= p3
    else:
        print("%s: skipping removing overscan levels from windowed data." % name)

def dotrim(name, header, data):
    binning = int(header["SDTBN"])
    yslice = slice(int(0 / binning), int(4096 / binning))
    xslice = slice(int(48 / binning), int(4144 / binning))
    return data[yslice, xslice]


In [ ]:
hdul = fits.open("20250221T033556C1o.fits.fz")
data = hdul[1].data.astype("float32")
header = hdul[1].header

plt.imshow(data, vmin=500, vmax=700)
plt.colorbar()
plt.show()

# Correct for overscan pedestal
dooverscan("sangrita", header, data)

plt.imshow(data, vmin=-200, vmax=200)
plt.colorbar()
plt.show()

# Trim image to the active regions
data = dotrim("sangrita", header, data)

plt.imshow(data, vmin=-200, vmax=200)
plt.colorbar()
plt.show()

# Subtract dark image

hdul = fits.open("dark-60.fits")
darkdata = hdul[0].data.astype("float32")
data -= darkdata

plt.imshow(data, vmin=-200, vmax=200)
plt.colorbar()
plt.show()

# Divide by flat image

hdul = fits.open("flat-i.fits")
flatdata = hdul[0].data.astype("float32")
data /= flatdata

plt.imshow(data, vmin=-200, vmax=200)
plt.colorbar()
plt.show()

In [ ]:
from astropy.stats import sigma_clipped_stats
mean, median, std = sigma_clipped_stats(data, sigma=3.0)
print(np.array((mean, median, std)))


In [ ]:
#data = data[1500:2500,1500:2500]
plt.figure(figsize=(10, 10))
zrange = 25
plt.imshow(data, cmap="gray_r", origin="lower", vmin=median-0.2*zrange, vmax=median+0.8*zrange)
plt.colorbar()
None

In [ ]:
! pip install astroscrappy
gain = 2.23
import astroscrappy
# mask cosmic rays using LACosmic algorithm

cmask, cdata = astroscrappy.detect_cosmics(data, None, gain=gain, verbose=True)
plt.figure(figsize=(10, 10))
zrange = 25
plt.imshow(cdata, cmap="gray_r", origin="lower", vmin=median-0.2*zrange, vmax=median+0.8*zrange)
plt.colorbar()
None


In [ ]:
from photutils.detection import DAOStarFinder
daofind = DAOStarFinder(fwhm=3.0, threshold=5.*std)
sources = daofind(data - median)
for col in sources.colnames:
    if col not in ('id', 'npix'):
        sources[col].info.format = '%.2f'  # for consistent table output
sources.pprint(max_width=76)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from photutils.aperture import CircularAperture

plt.figure(figsize=(10, 10))


positions = np.transpose((sources['xcentroid'], sources['ycentroid']))
apertures = CircularAperture(positions, r=10.0)
zrange = 25
plt.imshow(data, cmap="gray_r", origin="lower", vmin=median-0.2*zrange, vmax=median+0.8*zrange)
plt.colorbar()
apertures.plot(color='blue', lw=1.5, alpha=0.5)
None

In [ ]:
from photutils.aperture import CircularAnnulus, CircularAperture
from photutils.aperture import aperture_photometry
from photutils.aperture import ApertureStats

apertures = CircularAperture(positions, r=5.0)
background_apertures = CircularAnnulus(positions, r_in=10, r_out=15)

plt.figure(figsize=(20, 20))
plt.imshow(data, cmap="gray_r", origin="lower", vmin=median-0.2*zrange, vmax=median+0.8*zrange)
plt.colorbar()
apertures.plot(color='blue', lw=1.5, alpha=0.5)
background_apertures.plot(color='red', lw=1.5, alpha=0.5)

phot_table = aperture_photometry(data, apertures)
background_aperture_stats = ApertureStats(data, background_apertures)
background_mean = background_aperture_stats.mean

signal = phot_table['aperture_sum'] - apertures.area * background_mean

instrumental_magnitude = -2.5 * np.log10(signal) + 25

print(phot_table)
for i in range(len(phot_table)):
    print("%6.1f %6.1f %4.1f" % (phot_table['xcenter'][i], phot_table['ycenter'][i], instrumental_magnitude[i]))



#phot_table = aperture_photometry(data, apertures)


In [ ]:
col1 = fits.Column(name="x", format="E", array=phot_table['xcenter'])
col2 = fits.Column(name="y", format="E", array=phot_table['ycenter'])
col3 = fits.Column(name="mag", format="E", array=instrumental_magnitude)
coldefs = fits.ColDefs([col1, col2, col3])
hdu = fits.BinTableHDU.from_columns(coldefs)
hdu.writeto("phot.fits")